In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime
from delta.tables import DeltaTable

#### **transformation**

<mark>**DIM_PRODUCT**</mark>

In [ ]:
df_pro = spark.read.table('DimProduct')
df_cat = spark.read.table('DimProductCategory')
df_sub = spark.read.table('DimProductSubcategory')

In [ ]:
        # JOIN
# create table Subcategory + Category
df_mapping = df_sub.alias("s").join(df_cat.alias("c"), "ProductCategoryKey", "left")\
                    .select(
                            F.col("s.ProductSubcategoryKey"),
                            F.col("s.EnglishProductSubcategoryName").alias("SubCategoryName"),
                            F.col("c.EnglishProductCategoryName").alias("CategoryName")
                            )

# Join product table with mapping table
df_joined = df_pro.join(df_mapping, "ProductSubcategoryKey", "left")

        # CLEAN
# 
bad_words = ["Spanish", "French", "Chinese", "Arabic", "Hebrew", "Thai", "German", "Japanese", "Turkish"]

# 
cols_to_keep = [c for c in df_joined.columns if not any(word in c for word in bad_words)]
df_temp = df_joined.select(*cols_to_keep)

# 
df_final = df_temp.withColumn("ProductName", F.col("EnglishProductName")) \
                  .withColumn("Description", F.col("EnglishDescription")) \
                  .withColumn("Status", 
                                    F.when(F.col("Status").isNull() & F.col("EndDate").isNotNull(), "Discontinued")
                                    .when(F.col("Status").isNull(), "Active")
                                    .otherwise(F.col("Status"))) \
                  .withColumn("Color", 
                                    F.when((F.col("Color") == "NA") | (F.col("Color").isNull()), "Unknown")
                                    .otherwise(F.col("Color"))) \
                  .drop("EnglishProductName", "EnglishDescription")

        # WRITE
all_columns = df_final.columns
# 
ordered_columns = ["ProductKey"] + [c for c in all_columns if c != "ProductKey"]
df_final_reordered = df_final.select(*ordered_columns)
# 
df_final_reordered.write.format("delta").mode("overwrite").saveAsTable("silver_dimproduct")

display((df_final_reordered).limit(10))

**<mark>DIM_GEOGRAPHY</mark>**

In [ ]:
df_geo = spark.read.table('DimGeography')

# Loại bỏ các cột ngôn ngữ thừa
bad_words = ["Spanish", "French"]
cols_to_keep = [c for c in df_geo.columns if not any(word in c for word in bad_words)]

# Đổi tên cột
df_geo_silver = df_geo.select(*cols_to_keep) \
                        .withColumnRenamed("EnglishCountryRegionName", "Country") \
                        .withColumnRenamed("StateProvinceName", "State")

# Ghi ra silver
df_geo_silver.write.format("delta").mode("overwrite").saveAsTable("silver_DimGeography")

display(df_geo_silver.limit(5))

**<mark>DIM_CUSTOMER</mark>**

In [ ]:
df_cus = spark.read.table('DimCustomer')
# Clean
df_cus_clean = df_cus.withColumn(
                                "FullName", 
                                F.concat_ws(" ", F.col("FirstName"), F.col("MiddleName"), F.col("LastName")))\
                    .select(
                            "CustomerKey",
                            "GeographyKey",
                            "FullName",
                            "BirthDate",
                            "MaritalStatus",
                            "Gender",
                            "EmailAddress",
                            "YearlyIncome",
                            "TotalChildren",
                            "NumberCarsOwned",
                            F.col("EnglishEducation").alias("Education"),
                            F.col("EnglishOccupation").alias("Occupation"),
                            "AddressLine1",
                            "Phone",
                            "DateFirstPurchase")

# 3. Write
df_cus_clean.write.format("delta").mode("overwrite").saveAsTable("silver_DimCustomer")

display(df_cus_clean.limit(10))

**<mark>FACT_INTERNETSALES</mark>**

In [ ]:
# 1. Read data
df_fact = spark.read.table("FactInternetSales")

# 2. Transform
df_fact_silver = df_fact.select(
                                "ProductKey", "OrderDateKey", "CustomerKey", "SalesOrderNumber", "SalesOrderLineNumber",
                                "OrderQuantity", "UnitPrice", "ExtendedAmount", "SalesAmount", "TaxAmt", "Freight",
                                F.to_date("OrderDate").alias("OrderDate"),
                                F.to_date("ShipDate").alias("ShipDate"),
                                # revenue
                                F.round(F.col("SalesAmount") - F.col("TaxAmt") - F.col("Freight"), 2).alias("NetRevenue")
                            )

# 3. MERGE
if not spark.catalog.tableExists("silver_FactInternetSales"):
    # Tạo mới nếu chưa tồn tại
    df_fact_silver.write.format("delta").mode("overwrite").saveAsTable("silver_FactInternetSales")

else:
    # Upsert nếu tồn tại
    target_table = DeltaTable.forName(spark, "silver_FactInternetSales")
    # Key định danh: Số hóa đơn và Dòng hóa đơn
    join_condition = "target.SalesOrderNumber = source.SalesOrderNumber AND target.SalesOrderLineNumber = source.SalesOrderLineNumber"
    
    target_table.alias("target").merge(df_fact_silver.alias("source"),join_condition)\
                                .whenMatchedUpdateAll() \
                                .whenNotMatchedInsertAll() \
                                .execute()

display((df_fact_silver).limit(5))